In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
PROJECT_ROOT = Path("..").resolve()
DATASETS_ROOT = PROJECT_ROOT / "data/benchmark/resources/datasets"
FILES_ROOT = PROJECT_ROOT / "files"
OUTPUT_DATASET_ROOT = DATASETS_ROOT / "neurips-2023-data-subsample"


# 1. Get Fingerprints

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [ ]:
de_train = ad.read_h5ad(DATASETS_ROOT / "neurips-2023-data/de_train.h5ad")
de_test = ad.read_h5ad(DATASETS_ROOT / "neurips-2023-data/de_test.h5ad")

In [ ]:
de = ad.concat([de_train, de_test])
sm_smiles = de.obs[['sm_name', 'SMILES']].drop_duplicates()
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])
sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})

In [ ]:
sm_smiles

# 2. Load Pubchem

In [ ]:
#PubChemCIDs were obtained with the Chem-PerturBridge pipeline with the PubChemPy package
df_emb_op3 = pd.read_csv(FILES_ROOT / "df_pubchem_op3.csv")
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

# 3. Get embeddings and JOIN

In [ ]:
epoch_dirs = ['epoch_epoch_0019']

In [ ]:
epoch_dirs

In [ ]:
import os
from pathlib import Path
path = OUTPUT_DATASET_ROOT
os.makedirs(path, exist_ok=True)

for epoch_dir in epoch_dirs:
    
    df_pert_all = pd.read_pickle(FILES_ROOT / f"single_run_all_25_epochs/{epoch_dir}/df_pert.pkl")\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})
    

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    mask = df_emb_op3_merged['lpm_style_embeddings_all'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_all_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_all': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    df_emb_op3_all_to_save.to_pickle(f'{path}/op3_emb_all_{tag + 1}.pkl')

In [ ]:
for epoch_dir in epoch_dirs:
    
    df_pert_l1000 = pd.read_pickle(FILES_ROOT / f"single_run_l1000_25_epochs/{epoch_dir}/df_pert.pkl")\
                    .rename(columns={'symbol': 'symbol_l1000', 
                                     'code': 'code_l1000', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_l1000'})
    

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_l1000, left_on='pubchem_cid', right_on='symbol_l1000', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    mask = df_emb_op3_merged['lpm_style_embeddings_l1000'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_l1000_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_l1000': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    df_emb_op3_l1000_to_save.to_pickle(f'{path}/op3_emb_l1000_{tag + 1}.pkl')

In [ ]:
df_pert_all = pd.read_pickle(FILES_ROOT / f"single_run_all_25_epochs/{epoch_dir}/df_pert.pkl")\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})

    
df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                        .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

mask = df_emb_op3_merged['ECFP:2'].isna()
df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]

tag = int(epoch_dir.split('_')[-1])

df_emb_op3_all_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_all': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
df_emb_op3_all_to_save.to_pickle(f'{path}/op3_emb_fp.pkl')

# 4. Split

In [ ]:
ratio = 0.25

In [ ]:
op3_train_subsample = ad.read_h5ad(DATASETS_ROOT / "neurips-2023-data/de_train.h5ad")
op3_test_subsample = ad.read_h5ad(DATASETS_ROOT / "neurips-2023-data/de_test.h5ad")
df = pd.read_csv(DATASETS_ROOT / "neurips-2023-data/id_map.csv")

In [ ]:
op3_test_subsample

In [ ]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [ ]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [ ]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [ ]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [ ]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [ ]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad(OUTPUT_DATASET_ROOT / "de_train.h5ad", compression="gzip")

In [ ]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad(OUTPUT_DATASET_ROOT / "de_test.h5ad", compression="gzip")

In [ ]:
op3_test_subsample_.obs[["sm_name", "cell_type"]].reset_index(drop=True).reset_index().rename(columns={"index": "id"}).to_csv(OUTPUT_DATASET_ROOT / "id_map.csv", index=False)

In [ ]:
assert len(set(op3_test_subsample_.obs['sm_name']).intersection(set(op3_train_subsample_.obs['sm_name']))) == 0

In [ ]:
len(op3_test_subsample_.obs['sm_name'].unique()) + len(op3_train_subsample_.obs['sm_name'].unique())